# 🧱 Lab: Core Eval Pipeline

**Module 9: LLM Evaluation** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Build** an end-to-end evaluation pipeline that scores LLM outputs using multiple metric types
2. **Apply** reference-based metrics (BLEU, ROUGE) alongside LLM-as-judge scoring in a single workflow
3. **Create** a custom rubric tailored to a specific evaluation task and integrate it into the pipeline
4. **Analyze** pipeline results to identify quality patterns and decide which outputs pass or fail

## Concepts Covered

| Concept | From |
|---------|------|
| Why Evaluation Matters | mini-reference-eval |
| Reference-Based Evaluations (BLEU, ROUGE) | mini-reference-eval |
| LLM-as-Judge | mini-llm-judge |
| Rubric Scoring | mini-auto-grader |
| Automated Evaluation Pipeline | This lab |

## Prerequisites

- **mini-reference-eval**: Computing BLEU, ROUGE, and exact match scores
- **mini-llm-judge**: Using an LLM to judge subjective quality with pointwise and pairwise modes
- **mini-auto-grader**: Designing custom rubrics with weighted criteria and building an auto-grader
- OpenAI API key configured in `.env`

## Scenario: Evaluating a Q&A System

Imagine you've built a Q&A system that answers questions about a company's product documentation. Before deploying it, you need to **systematically evaluate** its outputs.

You have:
- A set of test questions with **reference answers** (written by domain experts)
- The **generated answers** from your Q&A system

Your evaluation pipeline will combine three layers:

| Layer | Method | What It Catches |
|-------|--------|-----------------|
| 1 | Reference-based (BLEU, ROUGE) | Missing content, wrong wording |
| 2 | LLM-as-Judge (pointwise) | Subjective quality — helpfulness, accuracy |
| 3 | Custom Rubric | Domain-specific criteria with weighted scoring |

No single layer is enough — together they give a complete picture.

## Setup

In [2]:
import os
import re
from dotenv import load_dotenv
import openai
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from IPython.display import display, Markdown

load_dotenv()
nltk.download('punkt_tab', quiet=True)

client = openai.OpenAI()

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

print("Setup complete.")

Setup complete.


## Evaluation Dataset

Five test cases from our product Q&A system. Each has a question, a reference answer (from documentation), and the system's generated answer. The outputs vary in quality — some are great, some have issues.

In [3]:
eval_dataset = [
    {
        "id": "Q1",
        "question": "What file formats does DataSync Pro support for import?",
        "reference": "DataSync Pro supports CSV, JSON, XML, and Parquet file formats for data import.",
        "generated": "DataSync Pro supports importing data from CSV, JSON, XML, and Parquet files."
    },
    {
        "id": "Q2",
        "question": "How do I reset my API key?",
        "reference": "To reset your API key, go to Settings > API Keys, click 'Regenerate', and confirm. The old key will be invalidated immediately.",
        "generated": "You can reset your API key from the Settings page. Look for the API section and regenerate it there."
    },
    {
        "id": "Q3",
        "question": "What is the maximum dataset size for the free tier?",
        "reference": "The free tier supports datasets up to 500 MB with a maximum of 1 million rows.",
        "generated": "Free tier users can work with datasets up to 500 MB and up to 1 million rows."
    },
    {
        "id": "Q4",
        "question": "Does DataSync Pro support scheduled exports?",
        "reference": "Yes, DataSync Pro supports scheduled exports. You can configure daily, weekly, or monthly export schedules in the Automation tab.",
        "generated": "DataSync Pro has some automation features. Check the docs for details on exports."
    },
    {
        "id": "Q5",
        "question": "How does row-level access control work?",
        "reference": "Row-level access control filters data based on user attributes. Admins define policies in Settings > Security that restrict which rows each user role can view.",
        "generated": "Row-level access control lets admins restrict data visibility by user role. Policies are configured in Settings > Security and filter rows based on user attributes, ensuring each user only sees data they are authorized to access."
    }
]

print(f"Evaluation dataset: {len(eval_dataset)} test cases")

Evaluation dataset: 5 test cases


## Layer 1: Reference-Based Metrics

Recall from `mini-reference-eval`: BLEU measures precision (how much of the generated text is in the reference) and ROUGE measures recall (how much of the reference is captured). We'll compute both for every test case.

In [4]:
smoother = SmoothingFunction().method1
rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

def compute_reference_metrics(reference, generated):
    """Compute BLEU and ROUGE scores for a single pair."""
    bleu = sentence_bleu(
        [reference.lower().split()],
        generated.lower().split(),
        smoothing_function=smoother
    )
    scores = rouge.score(reference, generated)
    return {
        "bleu": bleu,
        "rouge1_f1": scores['rouge1'].fmeasure,
        "rougeL_f1": scores['rougeL'].fmeasure
    }

In [5]:
# Run Layer 1 on all test cases
for item in eval_dataset:
    item["ref_metrics"] = compute_reference_metrics(item["reference"], item["generated"])

lines = [
    "### Layer 1: Reference-Based Metrics\n",
    "| ID | BLEU | ROUGE-1 (F1) | ROUGE-L (F1) |",
    "|----|------|-------------|-------------|"
]
for item in eval_dataset:
    m = item["ref_metrics"]
    lines.append(f"| {item['id']} | {m['bleu']:.3f} | {m['rouge1_f1']:.3f} | {m['rougeL_f1']:.3f} |")

md("\n".join(lines))

### Layer 1: Reference-Based Metrics

| ID | BLEU | ROUGE-1 (F1) | ROUGE-L (F1) |
|----|------|-------------|-------------|
| Q1 | 0.402 | 0.880 | 0.720 |
| Q2 | 0.055 | 0.450 | 0.350 |
| Q3 | 0.322 | 0.667 | 0.606 |
| Q4 | 0.026 | 0.312 | 0.187 |
| Q5 | 0.140 | 0.656 | 0.426 |

Reference metrics give us a quick, objective baseline. Notice Q4 likely scores low — the generated answer is vague and misses key details from the reference. But these metrics can't tell us *why* an answer is bad. That's what Layer 2 is for.

## Layer 2: LLM-as-Judge (Pointwise)

Recall from `mini-llm-judge`: we use an LLM to score each response on subjective criteria. Here we evaluate **Accuracy** (factual correctness given the reference) and **Completeness** (does it cover everything the reference covers?).

In [6]:
JUDGE_PROMPT = """You are an expert evaluator for a product Q&A system.

**Question:** {question}
**Reference Answer:** {reference}
**Generated Answer:** {generated}

Score the generated answer on these criteria (1-5 each):

- **Accuracy**: Is the generated answer factually correct compared to the reference?
- **Completeness**: Does the generated answer cover all the key information in the reference?

Format your answer EXACTLY like this:
Accuracy: <score>/5 - <justification>
Completeness: <score>/5 - <justification>
"""

def llm_judge(item, model="gpt-4o-mini"):
    """Score a single test case using LLM-as-judge."""
    prompt = JUDGE_PROMPT.format(
        question=item["question"],
        reference=item["reference"],
        generated=item["generated"]
    )
    result = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    judgment = result.choices[0].message.content
    
    scores = {}
    for criterion in ["Accuracy", "Completeness"]:
        match = re.search(rf"{criterion}:\s*(\d)/5\s*-\s*(.+)", judgment)
        if match:
            scores[criterion] = {
                "score": int(match.group(1)),
                "justification": match.group(2).strip()
            }
    return scores

print("LLM judge function ready.")

LLM judge function ready.


In [7]:
# Run Layer 2 on all test cases
for item in eval_dataset:
    item["judge_scores"] = llm_judge(item)

lines = [
    "### Layer 2: LLM-as-Judge Scores\n",
    "| ID | Accuracy | Completeness | Notes |",
    "|----|----------|-------------|-------|"
]
for item in eval_dataset:
    js = item["judge_scores"]
    acc = js.get("Accuracy", {})
    comp = js.get("Completeness", {})
    note = comp.get("justification", "")[:60] + "..." if len(comp.get("justification", "")) > 60 else comp.get("justification", "")
    lines.append(
        f"| {item['id']} | {acc.get('score', '?')}/5 | {comp.get('score', '?')}/5 | {note} |"
    )

md("\n".join(lines))

### Layer 2: LLM-as-Judge Scores

| ID | Accuracy | Completeness | Notes |
|----|----------|-------------|-------|
| Q1 | 5/5 | 5/5 | The generated answer includes all the key information from t... |
| Q2 | 4/5 | 3/5 | The generated answer does not cover all the key information ... |
| Q3 | 5/5 | 5/5 | The generated answer includes all key information from the r... |
| Q4 | 2/5 | 2/5 | The generated answer lacks specific details about the schedu... |
| Q5 | 5/5 | 5/5 | The generated answer covers all the essential information fo... |

Now we can see *why* certain answers are weak. The LLM judge explains that Q4's answer is vague and doesn't actually answer the question. This is information that BLEU and ROUGE alone can't provide.

## Layer 3: Custom Rubric Scoring

Recall from `mini-auto-grader`: rubrics formalize "what good looks like" with criteria, score levels, and weights. For a product Q&A system, we care about domain-specific dimensions that generic judging doesn't capture.

In [8]:
qa_rubric = {
    "name": "Product Q&A Quality",
    "criteria": [
        {
            "name": "Factual Accuracy",
            "description": "Are all stated facts correct with respect to the reference?",
            "weight": 0.4,
            "levels": {
                1: "Contains incorrect information or hallucinated details",
                2: "Mostly correct but has a notable factual error",
                3: "Correct but imprecise or ambiguous",
                4: "Accurate with only minor imprecisions",
                5: "Fully accurate — all facts match the reference"
            }
        },
        {
            "name": "Actionability",
            "description": "Can the user take action based on this answer without needing additional info?",
            "weight": 0.35,
            "levels": {
                1: "No actionable information provided",
                2: "Vague direction but missing key steps or details",
                3: "Somewhat actionable but user needs to figure out details",
                4: "Clear and actionable with minor gaps",
                5: "Fully actionable — user can follow the answer immediately"
            }
        },
        {
            "name": "Conciseness",
            "description": "Is the answer appropriately concise without unnecessary filler?",
            "weight": 0.25,
            "levels": {
                1: "Extremely verbose or mostly irrelevant content",
                2: "Too long with significant filler",
                3: "Acceptable length but could be tighter",
                4: "Concise with minimal excess",
                5: "Perfectly concise — every word earns its place"
            }
        }
    ]
}

print(f"Rubric: {qa_rubric['name']} — {len(qa_rubric['criteria'])} criteria, weights sum to {sum(c['weight'] for c in qa_rubric['criteria'])}")

Rubric: Product Q&A Quality — 3 criteria, weights sum to 1.0


In [9]:
def build_rubric_prompt(rubric, question, reference, generated):
    """Convert a rubric into a judge prompt."""
    criteria_text = ""
    for c in rubric["criteria"]:
        levels_text = "\n".join(f"      {k}: {v}" for k, v in c["levels"].items())
        criteria_text += (
            f"\n  - **{c['name']}** (weight: {c['weight']})\n"
            f"    {c['description']}\n"
            f"    Score levels:\n{levels_text}\n"
        )

    format_lines = "\n".join(
        f"{c['name']}: <score>/5 - <justification>" for c in rubric["criteria"]
    )

    return f"""You are an expert evaluator using a custom rubric.

**Question:** {question}
**Reference Answer:** {reference}
**Generated Answer:** {generated}

**Rubric: {rubric['name']}**
{criteria_text}
Score the generated answer on EACH criterion. Use the score levels above.

Format your answer EXACTLY like this:
{format_lines}
"""


def rubric_grade(rubric, item, model="gpt-4o-mini"):
    """Grade a test case using the rubric-based auto-grader."""
    prompt = build_rubric_prompt(
        rubric, item["question"], item["reference"], item["generated"]
    )
    result = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    judgment = result.choices[0].message.content

    scores = {}
    for c in rubric["criteria"]:
        match = re.search(rf"{c['name']}:\s*(\d)/5\s*-\s*(.+)", judgment)
        if match:
            scores[c["name"]] = {
                "score": int(match.group(1)),
                "justification": match.group(2).strip(),
                "weight": c["weight"]
            }

    weighted_total = sum(s["score"] * s["weight"] for s in scores.values())
    return {"scores": scores, "weighted_total": weighted_total}

print("Rubric grader ready.")

Rubric grader ready.


In [10]:
# Run Layer 3 on all test cases
for item in eval_dataset:
    item["rubric_result"] = rubric_grade(qa_rubric, item)

lines = [
    "### Layer 3: Custom Rubric Scores\n",
    "| ID | Factual Accuracy | Actionability | Conciseness | Weighted Total |",
    "|----|-----------------|--------------|-------------|---------------|"
]
for item in eval_dataset:
    r = item["rubric_result"]["scores"]
    fa = r.get("Factual Accuracy", {}).get("score", "?")
    ac = r.get("Actionability", {}).get("score", "?")
    co = r.get("Conciseness", {}).get("score", "?")
    wt = item["rubric_result"]["weighted_total"]
    lines.append(f"| {item['id']} | {fa}/5 | {ac}/5 | {co}/5 | **{wt:.2f}/5** |")

md("\n".join(lines))

### Layer 3: Custom Rubric Scores

| ID | Factual Accuracy | Actionability | Conciseness | Weighted Total |
|----|-----------------|--------------|-------------|---------------|
| Q1 | 5/5 | 5/5 | 5/5 | **5.00/5** |
| Q2 | 3/5 | 3/5 | 4/5 | **3.25/5** |
| Q3 | 5/5 | 5/5 | 5/5 | **5.00/5** |
| Q4 | 2/5 | 2/5 | 4/5 | **2.50/5** |
| Q5 | 5/5 | 4/5 | 5/5 | **4.65/5** |

## Combining All Layers: The Full Pipeline Report

Now we bring all three layers together into a single dashboard. This is the core value of a pipeline — instead of looking at one metric, you get a **multi-dimensional view** of each output's quality.

In [11]:
PASS_THRESHOLD = 3.5  # Weighted rubric score threshold for pass/fail

lines = [
    "## 📊 Full Pipeline Report\n",
    "| ID | BLEU | ROUGE-L | Judge Accuracy | Judge Completeness | Rubric Weighted | Pass? |",
    "|----|------|---------|---------------|-------------------|----------------|-------|"
]

pass_count = 0
for item in eval_dataset:
    rm = item["ref_metrics"]
    js = item["judge_scores"]
    wt = item["rubric_result"]["weighted_total"]
    passed = wt >= PASS_THRESHOLD
    if passed:
        pass_count += 1
    
    lines.append(
        f"| {item['id']} "
        f"| {rm['bleu']:.3f} "
        f"| {rm['rougeL_f1']:.3f} "
        f"| {js.get('Accuracy', {}).get('score', '?')}/5 "
        f"| {js.get('Completeness', {}).get('score', '?')}/5 "
        f"| {wt:.2f}/5 "
        f"| {'✅' if passed else '❌'} |"
    )

lines.append(f"\n**Pass rate: {pass_count}/{len(eval_dataset)} ({pass_count/len(eval_dataset)*100:.0f}%)** (threshold: {PASS_THRESHOLD}/5)")

md("\n".join(lines))

## 📊 Full Pipeline Report

| ID | BLEU | ROUGE-L | Judge Accuracy | Judge Completeness | Rubric Weighted | Pass? |
|----|------|---------|---------------|-------------------|----------------|-------|
| Q1 | 0.402 | 0.720 | 5/5 | 5/5 | 5.00/5 | ✅ |
| Q2 | 0.055 | 0.350 | 4/5 | 3/5 | 3.25/5 | ❌ |
| Q3 | 0.322 | 0.606 | 5/5 | 5/5 | 5.00/5 | ✅ |
| Q4 | 0.026 | 0.187 | 2/5 | 2/5 | 2.50/5 | ❌ |
| Q5 | 0.140 | 0.426 | 5/5 | 5/5 | 4.65/5 | ✅ |

**Pass rate: 3/5 (60%)** (threshold: 3.5/5)

## Drill-Down: Inspecting Failures

A pipeline is only useful if it helps you **find and fix problems**. Let's automatically surface the failing test cases with their rubric justifications so you know exactly what went wrong.

In [12]:
failures = [
    item for item in eval_dataset
    if item["rubric_result"]["weighted_total"] < PASS_THRESHOLD
]

if not failures:
    md("**All test cases passed!** No failures to inspect.")
else:
    lines = [f"### ❌ {len(failures)} Failing Test Case(s)\n"]
    for item in failures:
        lines.append(f"---\n")
        lines.append(f"**{item['id']}: {item['question']}**\n")
        lines.append(f"- **Reference:** {item['reference']}")
        lines.append(f"- **Generated:** {item['generated']}\n")
        lines.append(f"| Criterion | Score | Why |")
        lines.append(f"|-----------|-------|-----|")
        for name, data in item["rubric_result"]["scores"].items():
            lines.append(f"| {name} | {data['score']}/5 | {data['justification']} |")
        lines.append(f"\n**Weighted Total: {item['rubric_result']['weighted_total']:.2f}/5**\n")
    
    md("\n".join(lines))

### ❌ 2 Failing Test Case(s)

---

**Q2: How do I reset my API key?**

- **Reference:** To reset your API key, go to Settings > API Keys, click 'Regenerate', and confirm. The old key will be invalidated immediately.
- **Generated:** You can reset your API key from the Settings page. Look for the API section and regenerate it there.

| Criterion | Score | Why |
|-----------|-------|-----|
| Factual Accuracy | 3/5 | The generated answer is correct in stating that the API key can be reset from the Settings page, but it lacks specific details about the steps involved, such as the need to click 'Regenerate' and confirm, which are important for full accuracy. |
| Actionability | 3/5 | The answer provides some actionable information by indicating that the user can reset the API key from the Settings page, but it does not specify the exact steps needed to complete the action, leaving the user to figure out the details. |
| Conciseness | 4/5 | The answer is concise and does not contain unnecessary filler, but it could be slightly more informative without becoming verbose. Overall, it is clear and to the point. |

**Weighted Total: 3.25/5**

---

**Q4: Does DataSync Pro support scheduled exports?**

- **Reference:** Yes, DataSync Pro supports scheduled exports. You can configure daily, weekly, or monthly export schedules in the Automation tab.
- **Generated:** DataSync Pro has some automation features. Check the docs for details on exports.

| Criterion | Score | Why |
|-----------|-------|-----|
| Factual Accuracy | 2/5 | The generated answer mentions that DataSync Pro has automation features but does not confirm that it supports scheduled exports, which is a key detail from the reference answer. This omission constitutes a notable factual error. |
| Actionability | 2/5 | The answer suggests checking the documentation for details, but it lacks specific guidance on how to find or configure scheduled exports. This makes it vague and not fully actionable. |
| Conciseness | 4/5 | The answer is relatively concise and does not contain excessive filler, but it could be improved by providing more specific information about scheduled exports instead of directing the user to the documentation. |

**Weighted Total: 2.50/5**


## Wrapping It Up: The `evaluate_pipeline` Function

Let's package the entire 3-layer pipeline into a single reusable function. This is the pattern you'd use in a real project — run it nightly, after every model change, or before any deployment.

In [13]:
def evaluate_pipeline(dataset, rubric, pass_threshold=3.5, model="gpt-4o-mini"):
    """Run the full 3-layer evaluation pipeline on a dataset."""
    results = []
    
    for item in dataset:
        # Layer 1: Reference-based
        ref_metrics = compute_reference_metrics(item["reference"], item["generated"])
        
        # Layer 2: LLM-as-Judge
        judge_scores = llm_judge(item, model=model)
        
        # Layer 3: Custom Rubric
        rubric_result = rubric_grade(rubric, item, model=model)
        
        passed = rubric_result["weighted_total"] >= pass_threshold
        
        results.append({
            "id": item["id"],
            "question": item["question"],
            "ref_metrics": ref_metrics,
            "judge_scores": judge_scores,
            "rubric_result": rubric_result,
            "passed": passed
        })
    
    pass_rate = sum(1 for r in results if r["passed"]) / len(results)
    return {"results": results, "pass_rate": pass_rate}

print("evaluate_pipeline() ready — your reusable eval function.")

evaluate_pipeline() ready — your reusable eval function.


Let's verify it works by running it on a small subset (first 2 items) to confirm the output structure.

In [14]:
quick_test = evaluate_pipeline(eval_dataset[:2], qa_rubric)

lines = ["### Quick Pipeline Test (2 items)\n"]
for r in quick_test["results"]:
    lines.append(
        f"- **{r['id']}**: BLEU={r['ref_metrics']['bleu']:.3f}, "
        f"Rubric={r['rubric_result']['weighted_total']:.2f}/5, "
        f"{'✅ Pass' if r['passed'] else '❌ Fail'}"
    )
lines.append(f"\n**Pass rate: {quick_test['pass_rate']:.0%}**")

md("\n".join(lines))

### Quick Pipeline Test (2 items)

- **Q1**: BLEU=0.402, Rubric=5.00/5, ✅ Pass
- **Q2**: BLEU=0.055, Rubric=3.25/5, ❌ Fail

**Pass rate: 50%**

## Why Three Layers?

| Layer | Strengths | Weaknesses |
|-------|-----------|------------|
| Reference-based (BLEU, ROUGE) | Fast, deterministic, free | Can't judge quality — only word overlap |
| LLM-as-Judge | Understands meaning, explains reasoning | Costs API calls, may have biases |
| Custom Rubric | Domain-specific, weighted by business priority | Requires rubric design effort |

Together, they catch issues no single layer would find:
- Low ROUGE + high judge accuracy → answer is correct but uses very different phrasing
- High ROUGE + low actionability → answer overlaps with reference words but isn't useful
- All layers agree it's bad → strong signal for regression

## 🎯 Summary

### What You Built

You built an end-to-end evaluation pipeline that scores Q&A system outputs using three complementary layers — reference-based metrics (BLEU, ROUGE), LLM-as-judge pointwise scoring, and custom rubric-based grading — then combines them into a single pass/fail report with drill-down into failures.

### Key Takeaways

1. **Multi-layer evaluation catches more issues** — reference metrics, LLM judges, and rubrics each have blind spots; combining them gives a complete quality picture
2. **Reference-based metrics are a fast baseline** — BLEU and ROUGE are free and deterministic, making them ideal as a first-pass sanity check before spending API calls on LLM judging
3. **LLM-as-judge explains quality** — unlike numeric metrics, the judge provides justifications that tell you *why* an answer is good or bad
4. **Custom rubrics encode domain priorities** — weighted criteria ensure the pipeline measures what matters most for your specific use case
5. **A reusable pipeline enables regression tracking** — packaging evaluation into a single function lets you run it after every model change, prompt edit, or data update to catch quality regressions early